# Summary

Test calling a deployed agent

In [1]:
import os, sys
import json
import time

from vertexai import agent_engines

import vertexai

# Import libraries from the Agent Development Kit
from google.adk.agents import Agent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


utils_path = "../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication

# Set environment variables
dotenv_path = "../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()

# Initialize Vertex AI API once per session
vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
              location=os.environ["GOOGLE_CLOUD_LOCATION"],
              staging_bucket=os.environ["STAGING_BUCKET"])



## Retrieve a deployed agent

In [2]:
dir(agent_engines)

# get a list of agents
for agent in agent_engines.list():
    print(agent)

# Delete if needed
# agent_engines.delete(resource_name="projects/1062597788108/locations/us-central1/reasoningEngines/907343383519821824",
#                      force=True)


resource name: projects/1062597788108/locations/us-central1/reasoningEngines/4194584083407306752
resource name: projects/1062597788108/locations/us-central1/reasoningEngines/5338815048108212224


In [3]:
os.getenv("AGENT_ENGINE_ID2")

'projects/1062597788108/locations/us-central1/reasoningEngines/4194584083407306752'

In [7]:

# Retreive an ADK agent
# agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID2"))

# dir(agent_engine)
# Create a session
session = agent_engine.create_session(user_id="u_123")

# type(agent_engine)


In [8]:
type(agent_engine)
type(session)

dict

## Test the agent with one query

In [9]:
q1 = "What are LEAP exams?"
q2 = "What are the responsibilities of the board members of a California community college?"

test_result = agent_engine.stream_query(message=q2, user_id="U_123")

events = []
for event in test_result:
    events.append(event)



In [10]:
for event in events:
    print(type(event))
    print(event.keys())
    print(event['content'].keys())
    # print(event['content']['parts'])
    # print(event['grounding_metadata'])
    print(event['author'])
    print(event['actions'])
    print()




<class 'dict'>
dict_keys(['content', 'grounding_metadata', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
rag_assistant
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}

<class 'dict'>
dict_keys(['content', 'grounding_metadata', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
search_assistant
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}

<class 'dict'>
dict_keys(['content', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
synthesis_agent
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}



In [11]:
for event in events:
    if event["author"] == "synthesis_agent":
        for entry in event["content"]["parts"]:
            print(entry["text"])


California Community College board members have a wide array of responsibilities, primarily revolving around governance, policy-making, and ensuring the college's responsiveness to its community needs. These responsibilities include:

*   **Governance and Strategic Oversight:** Governing the community college district, adapting the district's mission to community needs, and ensuring college excellence and sustainability. This involves focusing on key issues related to fulfilling the district's mission.
*   **Educational Leadership:** Establishing educational priorities, and approving courses of instruction and educational programs.
*   **Executive Leadership:** Employing a district chief executive officer who recommends and implements local board policy and also hiring and evaluating the superintendent/president.
*   **Policy Formulation and Review:** Adopting policies that guide the operations of the district, and periodically reviewing the policy manual to ensure it is current and re

## Test the agent with multiple queries in a session

In [31]:
def pretty_print_event(event):
    """Pretty prints an event with truncation for long content."""
    if "content" not in event:
        print(f"[{event.get('author', 'unknown')}]: {event}")
        return

    author = event.get("author", "unknown")
    parts = event["content"].get("parts", [])

    for part in parts:
        if "text" in part:
            text = part["text"]
            # Truncate long text to 200 characters
            if len(text) > 200:
                text = text[:197] + "..."
            print(f"[{author}]: {text}")
        elif "functionCall" in part:
            func_call = part["functionCall"]
            print(f"[{author}]: Function call: {func_call.get('name', 'unknown')}")
            # Truncate args if too long
            args = json.dumps(func_call.get("args", {}))
            if len(args) > 100:
                args = args[:97] + "..."
            print(f"  Args: {args}")
        elif "functionResponse" in part:
            func_response = part["functionResponse"]
            print(f"[{author}]: Function response: {func_response.get('name', 'unknown')}")
            # Truncate response if too long
            response = json.dumps(func_response.get("response", {}))
            if len(response) > 100:
                response = response[:97] + "..."
            print(f"  Response: {response}")

agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
print(f"Agent ID: Id='{agent_engine.resource_name}'")

# Create a session
user_id = "u_123"
session = agent_engine.create_session(user_id=user_id)
print(f"Session created: User='{user_id}', Session='{session['id']}'")

# Create some queries
queries = [
    "Hi, how are you?",
    "What are LEAP exams?",
    "What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites"
]

# Look for events for each query
for query in queries:
    print(f"\n[user]: {query}")
    for event in agent_engine.stream_query(
        user_id=user_id,
        session_id=session["id"],
        message=query,
    ):
        print("We got here by finding an event")
        pretty_print_event(event)
        print(session.get(session["id"]))


Agent ID: Id='projects/1062597788108/locations/us-central1/reasoningEngines/5338815048108212224'
Session created: User='u_123', Session='5108327174356598784'

[user]: Hi, how are you?
We got here by finding an event
[ask_rag_agent]: I am doing well, thank you for asking! How are you today?

None

[user]: What are LEAP exams?
We got here by finding an event
[ask_rag_agent]: LEAP exams are an alternate examination and appointment process for the recruitment and hiring of individuals with disabilities into State service. Please contact the Department of Rehabilitation f...
None

[user]: What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites
We got here by finding an event
[ask_rag_agent]: The California Community Colleges Chancellor’s Office is committed to ensuring digital accessibility for people with disabilities by continually improving the user experience and applying the relev...
None


## Look at the sessions

In [7]:
agent_engine.list_sessions(user_id=user_id)

{'sessions': [{'events': [],
   'user_id': 'u_123',
   'state': {},
   'id': '8957849324595707904',
   'app_name': '5338815048108212224',
   'last_update_time': 1746668523.502969},
  {'events': [],
   'user_id': 'u_123',
   'state': {},
   'id': '3558033371378483200',
   'app_name': '5338815048108212224',
   'last_update_time': 1746668232.507558}]}

## Create - This stuff doesn't work with the deployed agent but does work if the deployed agent is called directly from code

In [7]:

def send_query_to_agent(agent, query, user_id):
    """Sends a query to the specified agent and prints the response.

    Args:
        agent: The agent to send the query to.
        query: The query to send to the agent.

    Returns:
        A tuple containing the elapsed time (in milliseconds) and the final response from the agent.
    """

    # Create a new session - if you want to keep the history of interruction you need to move the
    # creation of the session outside of this function. Here we create a new session per query
    session = session_service.create_session(app_name=AGENT_APP_NAME,
                                             user_id=user_id,)
    # Create a content object representing the user's query
    print('\nUser Query: ', query)
    content = types.Content(role='user', parts=[types.Part(text=query)])

    # Start a timer to measure the response time
    start_time = time.time()

    # Create a runner object to manage the interaction with the agent
    runner = Runner(app_name=AGENT_APP_NAME, agent=agent, artifact_service=artifact_service, session_service=session_service)

    # Run the interaction with the agent and get a stream of events
    events = runner.run(user_id=user_id, session_id=session.id, new_message=content)

    final_response = None
    elapsed_time_ms = 0.0

    # Loop through the events returned by the runner
    for _, event in enumerate(events):

        is_final_response = event.is_final_response()

        if not event.content:
             continue

        if is_final_response:
            end_time = time.time()
            elapsed_time_ms = round((end_time - start_time) * 1000, 3)

            print("-----------------------------")
            print('>>> Inside final response <<<')
            print("-----------------------------")
            final_response = event.content.parts[0].text # Get the final response from the agent
            print(f'Agent: {event.author}')
            print(f'Response time: {elapsed_time_ms} ms\n')
            print(f'Final Response:\n{final_response}')
            print("----------------------------------------------------------\n")

    return elapsed_time_ms, final_response



In [8]:
MODEL='gemini-2.0-flash-001'

# # Create a basic agent with instructions amd greeting only
# basic_agent = Agent(model=MODEL,
#     name="agent_basic",
#     description="This agent responds to inquiries about its creation by stating it was built using the Google Agent Framework.",
#     instruction="If they ask you how you were created, tell them you were created with the Google Agent Framework.",
#     generate_content_config=types.GenerateContentConfig(temperature=0.2),
# )

############## Import the agent code from the python file used to create the deployed agent
rag_path = "ccc_chatbot/sub_agents/rag"
sys.path.insert(0, rag_path)
from agent import root_agent

basic_agent = root_agent

# Get the agent
# basic_agent = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))

# Create session and memory services
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# AGENT_APP_NAME = basic_agent.resource_name
AGENT_APP_NAME = 'agent_basic'
user_id = "u_123"

# Send a single query to the agent
send_query_to_agent(basic_agent, "How are you today", user_id)



User Query:  How are you today
-----------------------------
>>> Inside final response <<<
-----------------------------
Agent: ask_rag_agent
Response time: 2493.828 ms

Final Response:
As an AI, I don't experience emotions in the same way humans do. However, I'm functioning optimally and ready to assist you with any questions you have about education policy.

----------------------------------------------------------



(2493.828,
 "As an AI, I don't experience emotions in the same way humans do. However, I'm functioning optimally and ready to assist you with any questions you have about education policy.\n")

In [9]:
dir(basic_agent)

# basic_agent.resource_name
# basic_agent.display_name

type(basic_agent)


google.adk.agents.llm_agent.LlmAgent